In [53]:
%use kandy

We are trying to estimate the performance ratio of stdlib functions in Kotlin/Native vs. JVM. For this example, we focus
on `String.substring` method used in this particular scenario. We need to exclude the effect of GC to get the performace
ratio of execution time of the function itself. The following cell describes, how we collect the data. We
end up with a list of execution times.

In [54]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardOpenOption

val kotlincNative = System.getProperty("user.home") + "/.konan/kotlin-native-prebuilt-linux-x86_64-2.2.10/bin/kotlinc-native"
val tempDir = Files.createTempDirectory("knative-perf")

fun runProcess(vararg command: String): String =
    ProcessBuilder(command.toList())
        .directory(tempDir.toFile())
        .start()
        .let { process ->
            process.inputStream.bufferedReader().readText()
                .also {
                    if (process.waitFor() != 0) {
                        process.errorStream.bufferedReader().readText().let(::println)
                        throw Exception("Process failed with code ${process.exitValue()}")
                    }
                }
        }

val source = """
import kotlin.concurrent.Volatile
import kotlin.time.measureTime
import kotlin.time.DurationUnit
import kotlin.time.Duration

@Volatile
private var blackhole: Any? = null
@Volatile
private var string = "abc".repeat(100000)

fun main() {
    repeat(1_000_000) { blackhole = string.substring(1000, 5000) }
    val iterations = 10_000
    val times = ArrayList<Duration>(iterations)
    repeat(iterations) {
        measureTime {
            repeat(100) {
                blackhole = string.substring(1000, 5000)
            }
        }.let { times.add(it) }
    }
    times.forEach { println(it.toDouble(DurationUnit.SECONDS)) }
}
"""

val sourceFile = tempDir.resolve("benchmark.kt")
val outputBinary = tempDir.resolve("benchmark.kexe")

Files.writeString(
    sourceFile,
    source,
    StandardOpenOption.CREATE,
)

runProcess(
        kotlincNative,
        sourceFile.toString(),
        "-o", outputBinary.toString(),
//        "-Xgc=noop"
)

val times = runProcess(outputBinary.toString())
    .split("\n")
    .mapNotNull { it.takeIf { it.isNotEmpty() }?.toDouble() }
    .toList()

For the exclusion of GC, we assume that the execution time distribution is bimodal (i .e. there are two peaks). The
first peak is formed by the executions that didn't trigger GC. The second peak is formed by the executions that
triggered GC. By finding a threshold for the execution time that lies between the two peaks, we can separate the
executions into those that ran GC and those which didn't.

In [55]:
val threshold = 0.00005

In [56]:
plot {
    histogram(times, binsOption = BinsOption.byNumber(100))
    vLine {
        xIntercept.constant(threshold)
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="Ik41ds"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("Ik41ds");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"x",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count"
},
"stat":"identity",
"data":{
"x":[1.568171E-5,4.704513E-5,7.840855E-5,1.0977197E-4,1.4113538999999998E-4,1.7249881E-4,2.0386223000000002E-4,2.3522565E-4,2.6658907E-4,2.9795249E-4,3.2931591E-4,3.6067933E-4,3.9204275000000004E-4,4.2340617E-4,4.5476958999999997E-4,4.8613301E-4,5.1749643E-4,5.4885985E-4,5.802232699999999E-4,6.1158669E-4,6.4295011E-4,6.7431353E-4,7.0567695E-4,7.3704037E-4,7.684037900000001E-4,7.9976721E-4,8.3113063E-4,8.6249405E-4,8.9385747E-4,9.2522089E-4,9.5658431E-4,9.8794773E-4,0.00101931115,0.00105067457,0.00108203799,0.00111340141,0.00114476483,0.0011761282500000001,0.00120749167,0.00123885509,0.00127021851,0.00130158193,0.00133294535,0.00136430877,0.00139567219,0.00142703561,0.00145839903,0.00148976245,0.00152112587,0.00155248929,0.00158385271,0.00161521613,0.00164657955,0.00167794297,0.00170930639,0.00174066981,0.00177203323,0.00180339665,0.00183476007,0.00186612349,0.00189748691,0.00192885033,0.00196021375,0.0019915771699999998,0.0020229405900000002,0.00205430401,0.0020856674300000003,0.00211703085,0.0021483942700000003,0.00217975769,0.0022111211100000003,0.00224248453,0.0022738479500000004,0.00230521137,0.00233657479,0.00236793821,0.0023993016299999996,0.00243066505,0.0024620284699999996,0.00249339189,0.0025247553099999997,0.00255611873,0.0025874821499999997,0.00261884557,0.0026502089899999998,0.0026815724100000002,0.00271293583,0.0027442992500000003,0.00277566267,0.0028070260900000003,0.00283838951,0.0028697529300000003,0.00290111635,0.0029324797700000004,0.00296384319,0.0029952066100000004,0.00302657003,0.0030579334499999996,0.00308929687,0.0031206602899999996],
"count":[7171.0,620.0,1068.0,667.0,275.0,114.0,40.0,13.0,3.0,4.0,6.0,4.0,4.0,1.0,1.0,1.0,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
}]
}
},{
"mapping":{
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"xintercept":5.0E-5,
"geom":"vline",
"data":{
}
}],
"spec_id":"32"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var

We then estimate the total time spent in GC by the following formula.

In [57]:
val (noGC, GC) = times.partition { it < threshold }
val avgNoGC = noGC.average()
GC.sumOf { it - avgNoGC } / times.sum()

0.4486028535588097

For measurements that run at least 100 000 iterations and don't print the times throught the execution (but all at once at the end), we estimate that GC is responsible for over 80% of the execution. That is insane. Therefore, we question the results.

* Can GC in K/N be responsible for such a high percentage of execution time? After all, this is very memory heavy benchmark.
* Is there an error in the benchmark setup, collection of the data or the processing?
* Is any one of our assumptions incorrect?
* Is this method for estimating GC time sound?